In [2]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


In [ ]:
from scripts.main import load_graph_and_communities

hg, comm = load_graph_and_communities("contact-primary-school", tau=None, verbose=True, source=False)

In [9]:
from scripts.main import load_graph_and_communities


for d in ['Algebra', 'Geometry', 'Music-Rev', 'Restaurants-Rev', 'Bars-Rev', 'contact-primary-school', 'contact-high-school']:
    hg, communities = load_graph_and_communities(d, tau=None, verbose=True, source=False)
    comm_nodes = set.union(*communities.values())

    print(f"Graph node type:     {type(list(hg.nodes)[0])}")
    print(f"Community node type: {type(list(comm_nodes)[0])}")
    print(f"Matching elements:   {len(hg.nodes.intersection(comm_nodes))}")
    print(len(hg.nodes), len(comm_nodes))

    print("Node in graph but missing from communities:", hg.nodes - comm_nodes)
    print("Node in communities but missing from graph:", comm_nodes - hg.nodes)
    # print(hg.nodes - set.union(*hg.hyperedges.values()))



Loaded dataset 'Algebra' (edge-labeled) with 1268 hyperedges and 32 communities
  communities sizes: min=2, max=167, avg=29.531
Graph node type:     <class 'str'>
Community node type: <class 'str'>
Matching elements:   423
423 423
Node in graph but missing from communities: set()
Node in communities but missing from graph: set()
Loaded dataset 'Geometry' (edge-labeled) with 1193 hyperedges and 25 communities
  communities sizes: min=2, max=374, avg=48.760
Graph node type:     <class 'str'>
Community node type: <class 'str'>
Matching elements:   580
580 580
Node in graph but missing from communities: set()
Node in communities but missing from graph: set()
Loaded dataset 'Music-Rev' (edge-labeled) with 694 hyperedges and 7 communities
  communities sizes: min=9, max=565, avg=171.429
Graph node type:     <class 'str'>
Community node type: <class 'str'>
Matching elements:   1106
1106 1106
Node in graph but missing from communities: set()
Node in communities but missing from graph: set()
Lo

In [ ]:
import csv
from h_louvain import hLouvain
from scripts.main import load_graph_and_communities_old_structure
# from core.data_loader import to_hypernetx
import hypernetx as hnx
import pandas as pd # Make sure this is at the top of your file!


datasets = ['contact-primary-school']

# Open a CSV file for writing (creates the file if it doesn't exist)
with open('modularity_scores.csv', mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    
    # Write the header row
    writer.writerow(['Dataset', 'Tau', 'Modularity_Score'])

    for d in datasets: 
        for tau in [0, 2]:
            graph_o, communities = load_graph_and_communities_old_structure(d, tau=tau, verbose=True, source=False)

            incidence_data = []
            for eid, nodes in graph_o.items():
                for node in nodes:
                    incidence_data.append((int(eid), str(node)))

            # 2. Build a pandas DataFrame
            df_incidence = pd.DataFrame(incidence_data, columns=["edge_id", "node_id"])

            # 3. Create the hypergraph explicitly telling HNX which columns to use
            hnxHG = hnx.Hypergraph(df_incidence, edge_col="edge_id", node_col="node_id")

            # hnxHG = to_hypernetx(hg)

            hnxComms = list(communities.values())

            hlv = hLouvain(hnxHG, hmod_tau=tau, resolution=1)

            partitions, mod, alphas = hlv.h_louvain_community()
            
            # Capture the modularity score instead of just calculating it
            mod_score = float(hlv.h_modularity(hnxComms, tau=tau, resolution=1))
            
            print(d)
            print(mod)
            print(mod_score)
            print()

            # # Write the current loop's data as a new row in the CSV
            # writer.writerow([d, tau, mod_score])
            
            # # Print to the console to track progress
            # print(f"Processed: {d} | tau: {tau} | Modularity: {mod_score:.4f}")

print("\nAll modularity scores successfully saved to 'modularity_scores.csv'.")

Loaded dataset 'contact-primary-school' (node-labeled) with 12704 hyperedges and 9 communities
  communities sizes: min=5, max=48, avg=26.889
contact-primary-school
0.3972875279129672
0.3964546970757596

Loaded dataset 'contact-primary-school' (node-labeled) with 12704 hyperedges and 6 communities
  communities sizes: min=24, max=51, avg=40.333
contact-primary-school
0.4214393141076618
0.42143931410766183


All modularity scores successfully saved to 'modularity_scores.csv'.


In [ ]:
import matplotlib.pyplot as plt
from scripts.main import load_graph_and_communities
from collections import Counter
import pandas as pd

datasets = ['Algebra', 'Geometry', 'Music-Rev', 'Restaurants-Rev', 'Bars-Rev', 'contact-primary-school', 'contact-high-school']

summary = []
dup_rows = []

for d in datasets:
    hg, communities = load_graph_and_communities(d, tau=2, verbose=True, source=False)
    unique_node_sets = set(frozenset(v) for v in hg.hyperedges.values())
    print(f"Dataset: {d}, len(hg): {len(hg.hyperedges)}, len(unique_node_sets): {len(unique_node_sets)}, duplicates: {len(hg.hyperedges) - len(unique_node_sets)}")

    edge_sets = [frozenset(e) for e in hg.hyperedges.values()]
    total = len(edge_sets)
    cnt = Counter(edge_sets)
    unique = len(cnt)
    duplicates = total - unique
    summary.append({
        "dataset": d,
        "total_hyperedges": total,
        "unique_hyperedge_sets": unique,
        "duplicate_hyperedges": duplicates,
        "duplicate_fraction": duplicates / total if total else 0,
    })
    for s, c in cnt.items():
        if c > 1:
            dup_rows.append({"dataset": d, "hyperedge": ",".join(sorted(s)), "count": c})

summary_df = pd.DataFrame(summary).sort_values("duplicate_hyperedges", ascending=False).reset_index(drop=True)
duplicates_df = pd.DataFrame(dup_rows).sort_values(["dataset", "count"], ascending=[True, False]).reset_index(drop=True)

display(summary_df)
if not duplicates_df.empty:
    display(duplicates_df)
else:
    print("No duplicate hyperedges found across datasets.")


In [ ]:
import statistics

import matplotlib.pyplot as plt

from scripts.main import load_graph_and_communities, run_configured_sicp_intermediates


hg,  communities = load_graph_and_communities("Geometry", tau=2, verbose=True, source=False)


BETA = 0.2
T = 25
RUNS = 10
RNG_SEED = 175
SEED_ITERATIONS = 200



edge_nodes = set().union(*hg.hyperedges.values()) if hg.hyperedges else set()
extra_nodes = hg.nodes - edge_nodes

res_old = run_configured_sicp_intermediates(
    hg,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)


N = len(hg.nodes)                      # total number of nodes in the simulation
timesteps = list(range(T + 1))
prevalence = [x / N for x in res_old]  # convert mean infected count -> mean prevalence

plt.figure(figsize=(8, 5))
plt.plot(timesteps, prevalence, marker="o")
plt.xlabel("Timestep")
plt.ylabel("Prevalence")
plt.title("SICP prevalence over time")
plt.ylim(0, 1)
plt.xlim(0, T)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path("../nkernokkel") / "Algebra_comprehensive_results.csv"
df = pd.read_csv(csv_path)

# Pick a time column if present; otherwise use row index
time_candidates = ["time", "timestep", "step", "t", "iteration"]
time_col = next((c for c in time_candidates if c in df.columns.str.lower()), None)

if time_col is None:
    # map lowercase match back to actual column names
    lower_to_actual = {c.lower(): c for c in df.columns}
    matched = next((c for c in time_candidates if c in lower_to_actual), None)
    time_col = lower_to_actual[matched] if matched else None

x = df[time_col] if time_col else df.index

# Plot all numeric columns except time
numeric_cols = df.select_dtypes(include="number").columns.tolist()
if time_col in numeric_cols:
    numeric_cols.remove(time_col)

# uses existing variables: df, col, time_col
agg = df.groupby(["strategy", "p", time_col])[col].mean().reset_index()

plt.figure(figsize=(10, 6))
for (strategy, p), g in agg.groupby(["strategy", "p"]):
    g = g.sort_values(time_col)
    plt.plot(g[time_col], g[col], marker="o", label=f"{strategy} | p={p}")

plt.xlabel(time_col)
plt.ylabel(col)
plt.title("Mean infected over time by strategy and removal fraction")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
